# Chapter 47: Human Pose Estimation & Landmark Detection
## Multimedia & Computer Vision Learning Hub — ITERA
**Author:** Aji Pamungkas Tri Nurcahyo
**Objective:** Memahami estimasi pose manusia 2D, membedah representasi target heatmap Gaussian, mendekode koordinat sendi secara sub-piksel, serta menghitung metrik evaluasi standar industri Object Keypoint Similarity (OKS).

### 1. Landasan Teori & Formulasi Heatmap Gaussian
Nilai intensitas heatmap pada koordinat $(i, j)$ untuk sendi di $(x_k, y_k)$:
$$\mathbf{H}_k(i, j) = \exp\left( -\frac{(i - x_k)^2 + (j - y_k)^2}{2\sigma^2} \right)$$

Metrik Object Keypoint Similarity (OKS) benchmark MS COCO:
$$\text{OKS} = \frac{\sum_k \exp\left( -\frac{d_k^2}{2 s^2 \kappa_k^2} \right) \delta(v_k > 0)}{\sum_k \delta(v_k > 0)}$$

In [ ]:
# 2. Setup Lingkungan & Dual-Engine Check
import math
import numpy as np
import sys

print(f"Python Version: {sys.version}")
print(f"NumPy Version: {np.__version__}")

HAS_TORCH = False
try:
    import torch
    import torch.nn as nn
    import torchvision.models.detection as detection
    HAS_TORCH = True
    print(f"PyTorch: {torch.__version__} (Device: {'cuda' if torch.cuda.is_available() else 'cpu'})")
except ImportError:
    print("INFO: PyTorch tidak terpasang di host lokal. Beroperasi dalam Mode Analisis Algoritmik NumPy.")

In [ ]:
# 3. Sintesis Heatmap Gaussian 2D & Dekode Koordinat Argmax
def generate_gaussian_heatmap(h, w, cx, cy, sigma=2.0):
    y_grid, x_grid = np.ogrid[:h, :w]
    d2 = (x_grid - cx)**2 + (y_grid - cy)**2
    return np.exp(-d2 / (2.0 * sigma**2))

def decode_peak(heatmap):
    idx = np.argmax(heatmap)
    y, x = np.unravel_index(idx, heatmap.shape)
    return int(x), int(y), float(heatmap[y, x])

hm = generate_gaussian_heatmap(64, 48, 24.0, 32.0, sigma=2.0)
px, py, conf = decode_peak(hm)
print(f"Bentuk Tensor Heatmap: {hm.shape}")
print(f"Puncak Terdekode: ({px}, {py}) dengan Keyakinan: {conf:.4f}")
assert (px, py) == (24, 32)
assert np.isclose(conf, 1.0)

In [ ]:
# 4. Kalkulasi Metrik Object Keypoint Similarity (OKS)
def compute_oks(pred_kpts, gt_kpts, scale, kappas):
    dists = np.linalg.norm(pred_kpts - gt_kpts, axis=-1)
    var = 2.0 * (scale**2) * (kappas**2)
    sims = np.exp(-(dists**2) / var)
    return float(np.mean(sims))

# Data dari contoh kuliah Section 3 (Hidung & Pergelangan Tangan, Skala = 200)
gt = np.array([[50.0, 40.0], [80.0, 200.0]])
pred = np.array([[52.0, 40.0], [84.0, 203.0]])
kappas = np.array([0.026, 0.062])

score_oks = compute_oks(pred, gt, scale=200.0, kappas=kappas)
print(f"Skor Object Keypoint Similarity (OKS): {score_oks:.4f} (Ekspektasi: 0.9253)")
assert np.isclose(score_oks, 0.9253, atol=1e-3)

In [ ]:
# 5. Real PyTorch Keypoint Estimation Workflow (Keypoint R-CNN Specification)
if HAS_TORCH:
    # Load pretrained Keypoint R-CNN ResNet-50 FPN (lisensi BSD-3)
    keypoint_model = detection.keypointrcnn_resnet50_fpn(weights=detection.KeypointRCNN_ResNet50_FPN_Weights.DEFAULT)
    keypoint_model.eval()
    print("Pretrained Keypoint R-CNN Model Successfully Loaded for 17 COCO Keypoints.")
    print(f"Total Model Parameters: {sum(p.numel() for p in keypoint_model.parameters()):,}")
    
    # Citra uji tiruan (1, 3, 256, 192)
    x_dummy = [torch.randn(3, 256, 192)]
    with torch.no_grad():
        preds = keypoint_model(x_dummy)
    print("Inferensi Keypoint Model Selesai. Kunci Output:", list(preds[0].keys()))
else:
    print("Panduan Alur Model Estimasi Pose Riil (Keypoint R-CNN / YOLO-Pose):")
    print("1. Pretrained Model: torchvision.models.detection.keypointrcnn_resnet50_fpn(weights=DEFAULT)")
    print("2. Input Tensor: List citra float [0, 1] dalam format (C, H, W).")
    print("3. Output Output: Dictionary memuat 'boxes', 'labels', 'scores', dan 'keypoints' (N, 17, 3).")
    print("4. Jalankan pada Google Colab T4 GPU untuk visualisasi kerangka sendi tubuh manusia.")

### 6. Analisis Hasil, Evaluasi & Rekomendasi
1. **Stabilitas Regresi Heatmap**: Memetakan sendi tubuh ke heatmap Gaussian 2D terbukti jauh lebih stabil secara numerik dibandingkan regresi koordinat langsung $(x, y)$.
2. **Karakteristik Metrik OKS**: OKS secara elegan mengoreksi distorsi evaluasi akibat perbedaan ukuran badan manusia melalui normalisasi skala objek $s = \sqrt{w \cdot h}$.
3. **Invariansi terhadap Skala**: Jika jarak eror piksel dan dimensi skala citra membesar secara proporsional, nilai OKS tetap deterministik dan tidak terdistorsi.